# 04c – DistilBERT results
Metrics and figures from the GPU run (`results/distilbert/`), using the shared evaluation code.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data import load_split, LABELS, RESULTS_DIR
from src.plotting import set_style, save, TEXT_SECONDARY
from src.evaluate import record_metrics, per_class_report, plot_confusion_matrix
from src.diagrams import draw_chain

set_style()
DIST_DIR = Path(os.environ.get("DISTILBERT_DIR", RESULTS_DIR / "distilbert"))
run = json.loads((DIST_DIR / "run_info.json").read_text())
MODEL = "distilbert" if not run["smoke_test"] else "distilbert_SMOKE"
assert not run["smoke_test"] or os.environ.get("ALLOW_SMOKE") == "1", "These are smoke-test outputs, not a real Colab run."

val_p, test_p = pd.read_csv(DIST_DIR / "preds_val.csv"), pd.read_csv(DIST_DIR / "preds_test.csv")
if not run["smoke_test"]:
    test, val = load_split("test"), load_split("val")
    assert (test_p.text.values == test.text.values).all() and (val_p.text.values == val.text.values).all(), "Predictions are not aligned with the saved splits"
{k: run[k] for k in ["model_name", "max_len", "epochs", "batch_size", "lr", "best_epoch", "trainable_params", "device"]} | {"train_minutes": round(run["train_seconds"] / 60, 1)}

## 1. Metrics

In [ ]:
extra = dict(params=run["trainable_params"], train_seconds=round(run["train_seconds"], 1))
record_metrics(MODEL, "val", val_p.label, val_p.pred, **extra)
table = record_metrics(MODEL, "test", test_p.label, test_p.pred, **extra)
test_p.to_csv(RESULTS_DIR / f"preds_{MODEL}.csv", index=False)   # same file layout as the other models
table[table.split == "test"].set_index("model")[["accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1", "params"]].round(4)

In [ ]:
rep = per_class_report(test_p.label, test_p.pred)
rep.to_csv(RESULTS_DIR / f"per_class_{MODEL}.csv")
cols = {"baseline": "baseline_tfidf_lr", "ANN": "ann_mlp", "BiLSTM": "bilstm"}
comp = pd.DataFrame({k: pd.read_csv(RESULTS_DIR / f"per_class_{v}.csv", index_col=0).f1 for k, v in cols.items()})
comp["DistilBERT"] = rep.f1
comp["DistilBERT − baseline"] = comp.DistilBERT - comp.baseline
comp.reindex(LABELS).round(3)

## 2. Architecture diagram

In [ ]:
chain = [("io", "Ticket text", "raw string"),
         ("prep", "WordPiece tokenizer", f"≤ {run['max_len']} sub-word tokens"),
         ("layer", "6 × Transformer layer", "self-attention, 12 heads"),
         ("prep", "[CLS] vector", "768 values"),
         ("layer", "Dense + ReLU", "768 units"),
         ("prep", "Dropout", "rate 0.2"),
         ("output", "Dense + Softmax", "8 support groups")]
fig = draw_chain(chain, "DistilBERT (fine-tuned Transformer) architecture", "18_distilbert_architecture",
                 subtitle=f"{run['trainable_params']:,} trainable parameters · pre-trained on English text, then fine-tuned on tickets · "
                          f"AdamW (lr {run['lr']}) · class-weighted cross-entropy")
plt.show()

## 3. Training curves

In [ ]:
h = pd.DataFrame(run["history"])
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, key, lab in [(axes[0], "loss", "Loss (cross-entropy)"), (axes[1], "accuracy", "Accuracy")]:
    ax.plot(h.epoch, h[key], color="#2a78d6", marker="o", markersize=6, linewidth=2, label="Training")
    ax.plot(h.epoch, h[f"val_{key}"], color="#eb6834", marker="s", markersize=6, linewidth=2, label="Validation")
    ax.axvline(run["best_epoch"], color=TEXT_SECONDARY, linestyle="--", linewidth=1)
    ax.set_xlabel("Epoch")
    ax.set_title(lab, fontsize=11)
    ax.set_xticks(h.epoch)
axes[0].legend(frameon=False, loc="lower left")
fig.suptitle("DistilBERT fine-tuning curves", x=0.01, ha="left", fontweight="bold", fontsize=13)
fig.tight_layout()
save(fig, "19_distilbert_training_curves")
plt.show()
h.round(4)

## 4. Confusion matrix

In [ ]:
fig, cm = plot_confusion_matrix(test_p.label, test_p.pred, "DistilBERT (fine-tuned) – test set", "20_distilbert_confusion_matrix")
plt.show()